# Test-Set Evaluation — Allen Human Depth-3 Model (44 classes)

Evaluate the trained depth-3 model on the held-out test donor **H0351.1015** (634 images).
The model was trained on 4 donors, validated on H0351.1016 (641 images), and this
notebook runs the final test evaluation on H0351.1015.

**Model source:** `/dbfs/FileStore/allen_brain_data/models/human-allen-depth3`

**Evaluation protocol:**
- Center-crop (CC): single 518×518 center crop per image
- Sliding window (SW): 518×518 tiles at stride 259 (50% overlap), resized to max 1024px
- SW runs on ALL 634 test images (not 50-image subset used during training validation)

**Output:** Test metrics logged to MLflow + per-class IoU saved as JSON for figure generation.

In [ ]:
# Cell 0 — Install project wheel from DBFS

%pip install /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
dbutils.library.restartPython()

In [ ]:
# Cell 1 — Configuration

import os
import mlflow

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"

EXPERIMENT_NAME = "/Users/noel.nosse@grainger.com/histology-human-brain-segmentation"
try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except Exception:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
mlflow.set_experiment(experiment_id=experiment_id)
print(f"MLflow experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")

# ---------- Databricks paths ----------
WORKSPACE_BASE = "/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology"
ONTOLOGY_PATH = f"{WORKSPACE_BASE}/ontology/structure_graph_10.json"
METADATA_PATH = f"{WORKSPACE_BASE}/metadata/human_atlas_images_metadata.json"
IMAGES_DIR = f"{WORKSPACE_BASE}/human_atlas/images"
SVGS_DIR = f"{WORKSPACE_BASE}/human_atlas/svgs"

# ---------- Saved model ----------
SAVED_MODEL_DIR = "/dbfs/FileStore/allen_brain_data/models/human-allen-depth3"

# ---------- Donor split ----------
TRAIN_DONORS = ["H0351.2002", "H0351.2001", "H0351.1012", "H0351.1009"]
VAL_DONORS = ["H0351.1016"]
TEST_DONORS = ["H0351.1015"]

# ---------- Depth-3 mapping ----------
TARGET_DEPTH = 3

# ---------- Eval config ----------
CROP_SIZE = 518
BATCH_SIZE = 2

# ---------- Pre-cache config ----------
CACHE_DIR = "/tmp/allen_cache_depth3_test"
CACHE_MAX_DIM = 1024

# ---------- Output ----------
RESULTS_DIR = SAVED_MODEL_DIR

print(f"{'='*65}")
print(f"TEST-SET EVALUATION: DEPTH-3 MODEL (44 CLASSES)")
print(f"{'='*65}")
print(f"Model:           {SAVED_MODEL_DIR}")
print(f"Test donor:      {TEST_DONORS}")
print(f"Cache:           {CACHE_DIR} (max {CACHE_MAX_DIM}px)")
print(f"Results dir:     {RESULTS_DIR}")

In [ ]:
# Cell 2 — Build test dataset with depth-3 mapping + pre-cache
#
# Reconstructs the same mapping pipeline used during training:
# 1. Scan TRAINING SVGs for structure IDs (to build identical present_mapping)
# 2. Build depth-3 present_mapping
# 3. Pre-cache TEST images only (634 files)
# 4. Create AllenHumanDataset for test split

import json
import time
from pathlib import Path
from xml.etree import ElementTree as ET

from histological_image_analysis.ontology import OntologyMapper
from histological_image_analysis.svg_rasterizer import SVGRasterizer
from histological_image_analysis.dataset import AllenHumanDataset, split_by_donor

# --- Step 1: Load metadata and build (image, svg, donor) triples ---
with open(METADATA_PATH) as f:
    metadata = json.load(f)

print(f"Total metadata entries: {len(metadata)}")

all_pairs = []
missing_images = 0
missing_svgs = 0

for entry in metadata:
    if not entry.get("annotated", False):
        continue

    donor_id = entry["_donor"]
    section_number = entry["section_number"]
    image_id = entry["id"]

    filename_base = f"{donor_id}_{section_number:04d}_{image_id}"
    image_path = Path(IMAGES_DIR) / f"{filename_base}.jpg"
    svg_path = Path(SVGS_DIR) / f"{filename_base}.svg"

    if not image_path.exists():
        missing_images += 1
        continue
    if not svg_path.exists():
        missing_svgs += 1
        continue

    all_pairs.append((image_path, svg_path, donor_id))

print(f"Annotated pairs found: {len(all_pairs)}")
if missing_images > 0:
    print(f"  Missing images: {missing_images}")
if missing_svgs > 0:
    print(f"  Missing SVGs: {missing_svgs}")

# --- Step 2: Split by donor ---
splits = split_by_donor(all_pairs, TRAIN_DONORS, VAL_DONORS, TEST_DONORS)
print(f"Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

# --- Step 3: Reconstruct the IDENTICAL mapping from training ---
# Must scan training SVGs to get the same structure ID set used during training.
# This ensures the present_mapping is identical to what the model was trained with.
print("Scanning training SVGs for structure IDs (to reconstruct training mapping)...")
all_structure_ids = set()
for img_path, svg_path in splits["train"]:
    tree = ET.parse(str(svg_path))
    root = tree.getroot()
    for elem in root.iter():
        sid_str = elem.get("structure_id", "")
        if sid_str:
            try:
                all_structure_ids.add(int(sid_str))
            except ValueError:
                pass

print(f"Unique structure IDs in training SVGs: {len(all_structure_ids)}")

# Alternative: load present_mapping.json from saved model (faster, same result)
mapping_json_path = os.path.join(SAVED_MODEL_DIR, "present_mapping.json")
if os.path.exists(mapping_json_path):
    with open(mapping_json_path) as f:
        present_mapping = {int(k): v for k, v in json.load(f).items()}
    print(f"Loaded present_mapping.json from saved model: {len(set(present_mapping.values()))} classes")
else:
    # Rebuild from scratch if JSON not saved
    mapper = OntologyMapper(ONTOLOGY_PATH)
    depth3_mapping = mapper.build_depth_mapping(target_depth=TARGET_DEPTH)
    present_class_ids = {
        depth3_mapping[sid]
        for sid in all_structure_ids
        if sid in depth3_mapping and depth3_mapping[sid] > 0
    }
    present_mapping = mapper.build_present_mapping(depth3_mapping, present_class_ids)
    print(f"Rebuilt present_mapping from scratch: {len(set(present_mapping.values()))} classes")

# Build mapper for class names
mapper = OntologyMapper(ONTOLOGY_PATH)
NUM_LABELS = mapper.get_num_labels(present_mapping)
class_names = mapper.get_class_names(present_mapping)
print(f"NUM_LABELS: {NUM_LABELS}")
print(f"Class names: {class_names[:10]}...")

# --- Step 4: Pre-cache TEST images ---
rasterizer = SVGRasterizer(mapper)

test_pairs = splits["test"]
print(f"\nPre-caching {len(test_pairs)} test images to {CACHE_DIR} (max {CACHE_MAX_DIM}px)...")
cache_start = time.time()
cached_count = AllenHumanDataset.build_cache(
    image_svg_pairs=test_pairs,
    rasterizer=rasterizer,
    mapping=present_mapping,
    cache_dir=CACHE_DIR,
    max_dim=CACHE_MAX_DIM,
)
cache_elapsed = time.time() - cache_start
print(f"Cached {cached_count} files in {cache_elapsed:.0f}s ({cache_elapsed/60:.1f} min)")

# --- Step 5: Create test dataset ---
test_ds = AllenHumanDataset(
    image_svg_pairs=test_pairs,
    rasterizer=rasterizer,
    mapping=present_mapping,
    crop_size=CROP_SIZE,
    augment=False,
    cache_dir=CACHE_DIR,
)
print(f"Test samples: {len(test_ds)}")

# Verify a sample
sample = test_ds[0]
print(f"pixel_values: {sample['pixel_values'].shape}, labels: {sample['labels'].shape}")
unique_labels = set(sample['labels'].numpy().ravel())
print(f"Unique labels in sample: {sorted(unique_labels)}")

In [ ]:
# Cell 3 — Load saved model from DBFS

import torch
from transformers import UperNetForSemanticSegmentation

print(f"Loading model from: {SAVED_MODEL_DIR}")
model = UperNetForSemanticSegmentation.from_pretrained(SAVED_MODEL_DIR)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Num labels (config): {model.config.num_labels}")
assert model.config.num_labels == NUM_LABELS, (
    f"Model num_labels ({model.config.num_labels}) != expected ({NUM_LABELS})"
)

# Forward pass validation
model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, CROP_SIZE, CROP_SIZE)
    if torch.cuda.is_available():
        model = model.cuda()
        dummy = dummy.cuda()
    out = model(pixel_values=dummy)
    print(f"Logits shape: {out.logits.shape}")
    assert out.logits.shape == (1, NUM_LABELS, CROP_SIZE, CROP_SIZE)

del out, dummy
torch.cuda.empty_cache()
model = model.cpu()
torch.cuda.empty_cache()
print("Forward pass OK — model loaded successfully")

In [ ]:
# Cell 4 — Center-crop evaluation on test set

import numpy as np
from histological_image_analysis.training import (
    get_training_args,
    make_compute_metrics,
    preprocess_logits_for_metrics,
)
from transformers import Trainer
from datetime import datetime

run_name = f"human-allen-depth{TARGET_DEPTH}-TEST-{datetime.now().strftime('%Y%m%d-%H%M')}"
mlflow.start_run(run_name=run_name)
mlflow.log_params({
    "track": f"A-depth{TARGET_DEPTH} TEST EVALUATION",
    "test_donor": ", ".join(TEST_DONORS),
    "test_samples": len(test_ds),
    "num_labels": NUM_LABELS,
    "model_dir": SAVED_MODEL_DIR,
    "crop_size": CROP_SIZE,
})

# Training args (eval-only — no training, just need Trainer infrastructure)
eval_args = get_training_args(
    output_dir="/tmp/eval_only",
    num_train_epochs=0,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    fp16=torch.cuda.is_available(),
    report_to="mlflow",
    dataloader_drop_last=False,
    load_best_model_at_end=False,
    save_strategy="no",
    eval_strategy="no",
)

trainer = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=test_ds,
    compute_metrics=make_compute_metrics(NUM_LABELS),
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)

# --- Center-crop evaluation ---
print(f"Running center-crop evaluation on {len(test_ds)} test images...")
eval_results = trainer.evaluate()

print("\nCenter-crop evaluation results:")
for k, v in sorted(eval_results.items()):
    if isinstance(v, float) and not k.startswith("eval_iou_class_"):
        print(f"  {k}: {v:.4f}")

cc_miou = eval_results.get('eval_mean_iou', 0.0)
cc_acc = eval_results.get('eval_overall_accuracy', 0.0)

mlflow.log_metrics({
    "test_cc_mean_iou": cc_miou,
    "test_cc_overall_accuracy": cc_acc,
})

# Per-class IoU
cc_class_ious = {}
for cls in range(NUM_LABELS):
    iou_key = f"eval_iou_class_{cls}"
    iou = eval_results.get(iou_key, float('nan'))
    if not np.isnan(iou):
        cc_class_ious[cls] = iou

sorted_ious = sorted(cc_class_ious.items(), key=lambda x: x[1], reverse=True)
print(f"\nTop 10 classes by IoU:")
for cls, iou in sorted_ious[:10]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

print(f"\nBottom 10 classes by IoU (non-zero):")
nonzero_ious = [(c, i) for c, i in sorted_ious if i > 0]
for cls, iou in nonzero_ious[-10:]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

print(f"\nClasses with valid IoU: {len(cc_class_ious)} / {NUM_LABELS}")
print(f"\nTest CC mIoU: {cc_miou:.1%}")
print(f"Test CC accuracy: {cc_acc:.1%}")

In [ ]:
# Cell 5 — Sliding window evaluation on ALL test images
#
# Runs SW eval on all 634 test images (not 50-image subset).
# This gives robust per-class IoU for the paper.

from PIL import Image
from tqdm.auto import tqdm

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
STRIDE = CROP_SIZE // 2  # 259 — 50% overlap
SW_MAX_DIM = 1024


def normalize_tile_rgb(tile):
    """uint8 RGB (H, W, 3) -> float32 tensor (1, 3, H, W)."""
    img = tile.astype(np.float32) / 255.0
    img_3ch = np.transpose(img, (2, 0, 1))  # (3, H, W)
    for c in range(3):
        img_3ch[c] = (img_3ch[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
    return torch.from_numpy(img_3ch).unsqueeze(0)


def sliding_window_predict_rgb(model, image, num_labels, crop_size, stride, device):
    """Predict full-resolution segmentation using overlapping tiles."""
    h, w = image.shape[:2]
    pad_h = max(0, crop_size - h)
    pad_w = max(0, crop_size - w)
    if pad_h > 0 or pad_w > 0:
        image = np.pad(image,
                       ((0, pad_h), (0, pad_w), (0, 0)),
                       mode='constant', constant_values=0)
    ph, pw = image.shape[:2]

    logit_sum = np.zeros((num_labels, ph, pw), dtype=np.float32)
    count_map = np.zeros((ph, pw), dtype=np.float32)

    y_starts = list(range(0, ph - crop_size + 1, stride))
    x_starts = list(range(0, pw - crop_size + 1, stride))
    if y_starts[-1] + crop_size < ph:
        y_starts.append(ph - crop_size)
    if x_starts[-1] + crop_size < pw:
        x_starts.append(pw - crop_size)

    use_cuda = device.type == "cuda"
    model.eval()
    for y in y_starts:
        for x in x_starts:
            tile = image[y:y + crop_size, x:x + crop_size]
            pixel_values = normalize_tile_rgb(tile).to(device)
            with torch.no_grad():
                if use_cuda:
                    with torch.amp.autocast("cuda", dtype=torch.float16):
                        logits = model(pixel_values=pixel_values).logits
                else:
                    logits = model(pixel_values=pixel_values).logits
            tile_logits = logits.squeeze(0).float().cpu().numpy()
            logit_sum[:, y:y + crop_size, x:x + crop_size] += tile_logits
            count_map[y:y + crop_size, x:x + crop_size] += 1.0

    count_map = np.maximum(count_map, 1.0)
    avg_logits = logit_sum / count_map[np.newaxis, :, :]
    pred = avg_logits.argmax(axis=0).astype(np.int64)
    return pred[:h, :w]


def resize_for_sliding_window(img_pil, mask, max_dim):
    """Resize image and mask so the longest side is max_dim pixels."""
    w, h = img_pil.size
    longest = max(h, w)
    if longest <= max_dim:
        return np.array(img_pil), mask, 1.0

    scale = max_dim / longest
    new_w = int(w * scale)
    new_h = int(h * scale)

    resized_img = img_pil.resize((new_w, new_h), resample=Image.BILINEAR)
    resized_mask = Image.fromarray(mask.astype(np.int32)).resize(
        (new_w, new_h), resample=Image.NEAREST
    )
    return np.array(resized_img), np.array(resized_mask).astype(np.int64), scale


# Run sliding window eval on ALL test images
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

test_pairs_for_sw = splits["test"]  # ALL 634 test images

all_sw_preds = []
all_sw_labels = []

print(f"\nSliding window eval: {len(test_pairs_for_sw)} test images, crop={CROP_SIZE}, stride={STRIDE}, max_dim={SW_MAX_DIM}")

# Build LUT for mask remapping
max_sid = max(max(present_mapping.keys()), 255)
lut = np.zeros(max_sid + 1, dtype=np.int64)
for sid, cid in present_mapping.items():
    if sid <= max_sid:
        lut[sid] = cid
lut[255] = 255

for img_path, svg_path in tqdm(test_pairs_for_sw, desc="Sliding window eval (test)"):
    # Load RGB image
    img_pil = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img_pil.size

    # Rasterize SVG mask at original image size
    raw_mask = rasterizer.rasterize(svg_path, target_width=orig_w, target_height=orig_h, sparse=True)

    # Remap
    class_mask = lut[raw_mask]

    # Resize for sliding window
    image_resized, mask_resized, scale = resize_for_sliding_window(img_pil, class_mask, SW_MAX_DIM)

    pred = sliding_window_predict_rgb(
        model, image_resized, NUM_LABELS, CROP_SIZE, STRIDE, device,
    )
    pred_h, pred_w = pred.shape
    mask_resized = mask_resized[:pred_h, :pred_w]

    # Only count labeled pixels (exclude 255)
    valid = mask_resized != 255
    if valid.sum() > 0:
        all_sw_preds.append(pred[valid].ravel())
        all_sw_labels.append(mask_resized[valid].ravel())

# Compute metrics
if all_sw_preds:
    preds_flat = np.concatenate(all_sw_preds)
    labels_flat = np.concatenate(all_sw_labels)

    sw_accuracy = float((preds_flat == labels_flat).sum()) / len(labels_flat)

    sw_class_ious = {}
    for cls in range(NUM_LABELS):
        label_mask = labels_flat == cls
        if label_mask.sum() == 0:
            continue
        pred_mask = preds_flat == cls
        intersection = (pred_mask & label_mask).sum()
        union = (pred_mask | label_mask).sum()
        sw_class_ious[cls] = float(intersection) / float(union) if union > 0 else 0.0

    sw_miou = float(np.mean(list(sw_class_ious.values()))) if sw_class_ious else 0.0
    sw_valid = len(sw_class_ious)

    print(f"\n{'='*65}")
    print(f"TEST SET EVALUATION RESULTS")
    print(f"{'='*65}")
    print(f"  Center-crop mIoU:        {cc_miou:.1%}")
    print(f"  Center-crop accuracy:    {cc_acc:.1%}")
    print(f"  Sliding window mIoU:     {sw_miou:.1%} ({len(test_pairs_for_sw)} images, resized to max {SW_MAX_DIM}px)")
    print(f"  Sliding window accuracy: {sw_accuracy:.1%}")
    print(f"  SW valid classes:        {sw_valid}")
    print(f"  CC valid classes:        {len(cc_class_ious)}")
    print(f"  mIoU delta (SW - CC):    {sw_miou - cc_miou:+.1%}")

    # SW per-class breakdown
    sw_sorted = sorted(sw_class_ious.items(), key=lambda x: x[1], reverse=True)
    print(f"\nSW Top 10 classes by IoU:")
    for cls, iou in sw_sorted[:10]:
        name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
        print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

    print(f"\nSW Bottom 10 classes by IoU (non-zero):")
    sw_nonzero = [(c, i) for c, i in sw_sorted if i > 0]
    for cls, iou in sw_nonzero[-10:]:
        name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
        print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

    # Log to MLflow
    mlflow.log_metrics({
        "test_sw_mean_iou": sw_miou,
        "test_sw_overall_accuracy": sw_accuracy,
        "test_sw_valid_classes": sw_valid,
        "test_sw_num_images": len(test_pairs_for_sw),
        "test_cc_valid_classes": len(cc_class_ious),
    })

    # --- Save results JSON ---
    results = {
        "test_donor": TEST_DONORS,
        "test_images": len(test_pairs_for_sw),
        "num_labels": NUM_LABELS,
        "class_names": class_names,
        "cc_mean_iou": cc_miou,
        "cc_overall_accuracy": cc_acc,
        "cc_valid_classes": len(cc_class_ious),
        "cc_per_class_iou": {class_names[cls]: iou for cls, iou in cc_class_ious.items()},
        "sw_mean_iou": sw_miou,
        "sw_overall_accuracy": sw_accuracy,
        "sw_valid_classes": sw_valid,
        "sw_num_images": len(test_pairs_for_sw),
        "sw_per_class_iou": {class_names[cls]: iou for cls, iou in sw_class_ious.items()},
    }

    results_path = os.path.join(RESULTS_DIR, "test_results.json")
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to: {results_path}")

    # Log as MLflow artifact
    mlflow.log_artifact(results_path, artifact_path="test_results")
    print("Results logged to MLflow as artifact")

else:
    print("No valid sliding window predictions (all masks empty?)")

model = model.cpu()
torch.cuda.empty_cache()

# Close MLflow run
mlflow.end_run()
print(f"\nMLflow run closed: {run_name}")